# Daily Challenge: Preprocess & Fine-tune Transformer-based Models

Guided notebook for preprocessing and fine-tuning **BERT** and **XLM-RoBERTa** for text classification.
Cells tagged **PREFILLED** run as-is; cells tagged **To-Do** are where you write/adjust code.

We use a **Natural Language Inference (NLI)** task (premise + hypothesis -> 3 classes:
`0 = entailment`, `1 = neutral`, `2 = contradiction`). This is the classic *"Contradictory, My Dear Watson"* setup,
which is exactly why we need **two-sentence tokenization** and a multilingual model like XLM-RoBERTa.


## Learning objectives
- Understand how BERT and XLM-RoBERTa process text via tokenization.
- Tokenize single sentences and sentence pairs with `encode_plus()` and inspect `input_ids`, `attention_mask`, `token_type_ids`.
- Format inputs correctly: padding, truncation, `max_length`, and special tokens.
- Load and explore the dataset with pandas.
- Build 5 stratified cross-validation folds with `StratifiedKFold`.


# Part 0: Setup

**PREFILLED: run once.** Installs the libraries used in this challenge.

In [ ]:
%pip install --quiet "transformers[sentencepiece]" scikit-learn pandas

**PREFILLED.** Core imports.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from transformers import BertTokenizer, XLMRobertaTokenizer

pd.set_option('display.max_colwidth', 60)
print('Imports OK')

# Part 1: Understanding BERT and XLM-RoBERTa

**BERT** (Bidirectional Encoder Representations from Transformers) is an encoder-only transformer pre-trained
on English text with *masked language modeling* and *next-sentence prediction*. It uses **WordPiece** tokenization
and the special tokens `[CLS]` (prepended; its final hidden state is used for classification) and `[SEP]`
(separates two sentences / marks the end). For sentence pairs it also produces `token_type_ids` (segment 0 vs 1).

**XLM-RoBERTa** is a *multilingual* RoBERTa trained on 100 languages with **SentencePiece** tokenization.
It drops next-sentence prediction and uses different special tokens: `<s>` (start, like `[CLS]`), `</s>` (separator/end),
`<pad>`, `<unk>`, `<mask>`. Because it is multilingual it is ideal when premises/hypotheses appear in many languages.

**Common pre-trained checkpoints**

| Model | Checkpoint | Tokenizer | Vocab | Notes |
|-------|-----------|-----------|-------|-------|
| BERT | `bert-base-uncased` | WordPiece | ~30k | English, lowercased |
| BERT | `bert-base-multilingual-cased` | WordPiece | ~120k | 104 languages |
| XLM-R | `xlm-roberta-base` | SentencePiece | ~250k | 100 languages |
| XLM-R | `xlm-roberta-large` | SentencePiece | ~250k | bigger, stronger |


**To-Do (code).** Load both pre-trained tokenizers.

In [ ]:
# TODO: load a BERT tokenizer and an XLM-RoBERTa tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

print('BERT  vocab size :', bert_tokenizer.vocab_size)
print('XLM-R vocab size :', xlmr_tokenizer.vocab_size)
print('BERT  special map:', bert_tokenizer.special_tokens_map)
print('XLM-R special map:', xlmr_tokenizer.special_tokens_map)

# Part 2: Tokenizing Text

`encode_plus()` turns raw text into the dictionary the model expects. Key outputs:
- **`input_ids`** — integer id of every token (special tokens included).
- **`attention_mask`** — `1` for real tokens, `0` for padding (so the model ignores padding).
- **`token_type_ids`** — *BERT only*: segment id (`0` = first sentence, `1` = second). XLM-R does not use these.
- **`labels`** — not produced by the tokenizer; you attach the class label yourself for training.


**To-Do (code).** Tokenize a **single sentence** with BERT and decode it back.

In [ ]:
sentence = 'Transformers make multilingual NLP much easier.'

# TODO: tokenize the single sentence
enc = bert_tokenizer.encode_plus(sentence, add_special_tokens=True, return_tensors=None)

print('input_ids     :', enc['input_ids'])
print('attention_mask:', enc['attention_mask'])
print('tokens        :', bert_tokenizer.convert_ids_to_tokens(enc['input_ids']))
print('decoded       :', bert_tokenizer.decode(enc['input_ids']))

**To-Do (code).** Tokenize a **sentence pair** (premise + hypothesis) with BERT.
Notice `token_type_ids` switches from 0 to 1 at the second sentence, and `[SEP]` appears twice.

In [ ]:
premise    = 'A man is playing a guitar on stage.'
hypothesis = 'A person is performing music.'

# TODO: tokenize the pair (pass both arguments)
pair = bert_tokenizer.encode_plus(premise, hypothesis, add_special_tokens=True)

print('tokens         :', bert_tokenizer.convert_ids_to_tokens(pair['input_ids']))
print('token_type_ids :', pair['token_type_ids'])
print('attention_mask :', pair['attention_mask'])

**To-Do (code).** Tokenize the same pair with **XLM-RoBERTa** and compare.
XLM-R wraps with `<s> ... </s></s> ... </s>` and returns **no** `token_type_ids`.

In [ ]:
xpair = xlmr_tokenizer.encode_plus(premise, hypothesis, add_special_tokens=True)

print('tokens :', xlmr_tokenizer.convert_ids_to_tokens(xpair['input_ids']))
print('keys   :', list(xpair.keys()))
print('decoded:', xlmr_tokenizer.decode(xpair['input_ids']))

# Part 3: Preparing Input Data for the Model

For batching, every example must have the **same length**. We `padding='max_length'` short sequences and
`truncation=True` long ones to a fixed `max_length`. The `attention_mask` then tells the model which positions
are real (`1`) vs padding (`0`).


**PREFILLED.** Inspect XLM-R special tokens and vocabulary size.

In [ ]:
print('special_tokens_map:', xlmr_tokenizer.special_tokens_map)
print('bos (<s>)  id:', xlmr_tokenizer.bos_token, xlmr_tokenizer.bos_token_id)
print('eos (</s>) id:', xlmr_tokenizer.eos_token, xlmr_tokenizer.eos_token_id)
print('pad        id:', xlmr_tokenizer.pad_token, xlmr_tokenizer.pad_token_id)
print('vocab_size   :', xlmr_tokenizer.vocab_size)

**To-Do (code).** Encode a pair with fixed `max_length`, padding and truncation.

In [ ]:
MAX_LEN = 32

# TODO: encode with padding='max_length' and truncation=True
fixed = xlmr_tokenizer.encode_plus(
    premise, hypothesis,
    add_special_tokens=True,
    max_length=MAX_LEN,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
)

print('length         :', len(fixed['input_ids']), '(should equal MAX_LEN)')
print('input_ids      :', fixed['input_ids'])
print('attention_mask :', fixed['attention_mask'])
print('# real tokens  :', sum(fixed['attention_mask']))
print('# pad tokens   :', len(fixed['attention_mask']) - sum(fixed['attention_mask']))

**PREFILLED.** A small helper that encodes a whole batch of pairs at once — exactly what you would feed
to the model during training/validation.

In [ ]:
def encode_pairs(tokenizer, premises, hypotheses, max_len=MAX_LEN):
    return tokenizer(
        list(premises), list(hypotheses),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='np',
    )

demo = encode_pairs(xlmr_tokenizer, [premise, 'Hello world'], [hypothesis, 'Bonjour le monde'])
print('batch input_ids shape     :', demo['input_ids'].shape)
print('batch attention_mask shape:', demo['attention_mask'].shape)

# Part 4: Loading and Exploring the Dataset

The dataset has the NLI columns `premise`, `hypothesis`, `label` (and usually `language`/`lang_abv`).
Place `train.csv` / `test.csv` in this folder. If they are not found, the next cell builds a small
**inline sample** so the rest of the notebook still runs end-to-end.

**PREFILLED.** Load `train.csv`/`test.csv` if present, otherwise fall back to an inline sample.

In [ ]:
from pathlib import Path

DATA_DIR = Path('.')
train_path, test_path = DATA_DIR / 'train.csv', DATA_DIR / 'test.csv'

if train_path.exists():
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path) if test_path.exists() else None
    print('Loaded train.csv from disk.')
else:
    print('train.csv not found -> using an inline sample dataset.')
    samples = [
        ('A man inspects a uniform.', 'The man is sleeping.', 2),
        ('A soccer game with players.', 'Some men are playing a sport.', 0),
        ('A smiling woman holds a baby.', 'A woman is at a meeting.', 1),
        ('Two dogs run in the park.', 'Animals are outdoors.', 0),
        ('The child is crying loudly.', 'The kid is completely silent.', 2),
        ('A chef cooks in a kitchen.', 'Someone is preparing food.', 0),
        ('People wait for the train.', 'The platform might be busy.', 1),
        ('A car is parked on the road.', 'There is no vehicle anywhere.', 2),
        ('She reads a book by the window.', 'A person is reading.', 0),
        ('The team won the final match.', 'The result of the game is unknown.', 1),
        ('Rain falls on the empty street.', 'The street is full of people.', 2),
        ('A boy kicks a football.', 'A child plays with a ball.', 0),
    ]
    reps = 30  # repeat to give StratifiedKFold enough rows per class
    rows = samples * reps
    train_df = pd.DataFrame(rows, columns=['premise', 'hypothesis', 'label'])
    train_df.insert(0, 'id', [f'{i:05d}' for i in range(len(train_df))])
    train_df['language'] = 'English'
    test_df = None

print('train shape:', train_df.shape)

**To-Do (code).** Explore the structure: `head()`, `shape`, columns, and the label distribution.

In [ ]:
# TODO: display the first rows
display(train_df.head())

print('shape       :', train_df.shape)
print('columns     :', list(train_df.columns))
print('label counts:')
print(train_df['label'].value_counts().sort_index())

# Part 5: Creating Cross-Validation Folds

With `StratifiedKFold(n_splits=5, shuffle=True)` we split into 5 train/validation folds while keeping the
**same label distribution** in every fold. We store each fold's row indices so we can train/evaluate per fold.

**To-Do (code).** Build 5 stratified folds and store the train/val index arrays.

In [ ]:
N_SPLITS = 5
X = train_df.index.values
y = train_df['label'].values

# TODO: create the StratifiedKFold object (shuffle=True, fixed random_state)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

train_folds, val_folds = [], []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_folds.append(train_idx)
    val_folds.append(val_idx)
    dist = train_df.iloc[val_idx]['label'].value_counts(normalize=True).sort_index().round(3).to_dict()
    print(f'Fold {fold}: train={len(train_idx):4d}  val={len(val_idx):4d}  val label ratio={dist}')

print('\nStored', len(train_folds), 'train/val folds.')

**To-Do (code).** Sanity check: tokenize fold 0's training split into model-ready tensors.

In [ ]:
fold0_train = train_df.iloc[train_folds[0]]

encoded = encode_pairs(xlmr_tokenizer, fold0_train['premise'], fold0_train['hypothesis'])
labels  = fold0_train['label'].values

print('input_ids      :', encoded['input_ids'].shape)
print('attention_mask :', encoded['attention_mask'].shape)
print('labels         :', labels.shape)
print('Ready to feed (input_ids, attention_mask, labels) into the model.')

# (Optional) Next step: fine-tuning

With the folds and the `encode_pairs` helper above, fine-tuning is straightforward. Sketch:

```python
from transformers import (XLMRobertaForSequenceClassification,
                          TrainingArguments, Trainer)
import torch

class NLIDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(int(self.labels[i]))
        return item

model = XLMRobertaForSequenceClassification.from_pretrained(
    'xlm-roberta-base', num_labels=3)

# loop over train_folds / val_folds, build NLIDataset for each, and train with Trainer
```

## Recap
- BERT (WordPiece, `[CLS]`/`[SEP]`, `token_type_ids`) vs XLM-RoBERTa (SentencePiece, `<s>`/`</s>`, multilingual).
- `encode_plus()` produces `input_ids`, `attention_mask`, and (BERT) `token_type_ids`; `decode()` reverses it.
- Padding + truncation to a fixed `max_length` make batching possible; the attention mask hides padding.
- `StratifiedKFold` gives 5 label-balanced folds for robust validation.
